# Modelos: regresión logística, árbol de decisión, KNN y MLP

**Entrada:** `dataset_final.csv` y `diccionario_dataset_final.csv` (cuaderno 03) y `particion.csv` (cuaderno 04). **Salida:** tablas de resultados (`resultados_cv.csv`, `resultados_prueba.csv`), las predicciones sobre la prueba (`predicciones_prueba.csv`) y los mejores parámetros.

## Cómo se compara (y por qué así)
Los cuatro modelos se entrenan igual y se evalúan con las mismas reglas, para que la comparación sea justa.

1. **Preprocesamiento dentro de cada pliegue.** El preprocesador (imputar, One-Hot, escalar) va dentro de un `Pipeline`: en cada vuelta de la validación cruzada se ajusta solo con los pliegues de entrenamiento de esa vuelta. Así ningún dato del pliegue de validación influye en cómo se prepara.
2. **Elegir hiperparámetros con la validación cruzada por conglomerado.** Cada combinación se prueba con los 5 pliegues. El criterio de elección es la **sensibilidad al 30 % de cobertura**: si el programa solo puede atender al 30 % de los hogares con mayor puntaje, ¿qué proporción de los hogares pobres queda dentro? Es el criterio más cercano al objetivo del trabajo (reducir el error de exclusión).
3. **Desbalance.** Regresión logística y árbol usan `class_weight="balanced"`. El MLP usa pesos por hogar equivalentes. KNN no admite pesos de clase: su desbalance se maneja con el umbral de decisión (paso 4).
4. **Umbral de decisión.** Se elige con las predicciones de validación cruzada (las de cada hogar cuando estuvo fuera del entrenamiento): el umbral que maximiza F1. Sirve para reportar sensibilidad, precisión y F1 "a umbral".
5. **Evaluación final, una sola vez, en prueba.** Se reportan: sensibilidad, precisión y F1 con ese umbral, y las métricas a cobertura fija (20, 30 y 40 %): error de exclusión (= 1 - sensibilidad: pobres que quedan fuera) y error de inclusión (= 1 - precisión: no pobres seleccionados).

**Desempates:** varios hogares pueden tener el mismo puntaje (por ejemplo, en el árbol). Al elegir el 30 % superior, el orden entre empatados se decide al azar con semilla fija, para no favorecer a nadie por su posición en la tabla.

In [ ]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, make_scorer, precision_recall_curve
from sklearn.model_selection import GridSearchCV, PredefinedSplit, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

RAIZ = Path.cwd().resolve().parent
PROCESSED = RAIZ / "data" / "processed"
SALIDA = PROCESSED / "resultados"
SALIDA.mkdir(parents=True, exist_ok=True)

RAPIDO = False            # True = cuadrículas pequeñas, para probar que todo corre (tarda minutos menos)
SEMILLA = 42
COBERTURAS = [0.20, 0.30, 0.40]
COBERTURA_AJUSTE = 0.30   # criterio para elegir hiperparámetros
N_JOBS = -1               # usa todos los núcleos; pon 1 si el equipo se pone lento


# Los identificadores se leen como texto para no perder los ceros a la izquierda.
datos = pd.read_csv(PROCESSED / "dataset_final.csv", encoding="utf-8-sig",
                    dtype={"id_hogar": str, "conglome": str, "ubigeo": str, "departamento": str})
particion = pd.read_csv(PROCESSED / "particion.csv", encoding="utf-8-sig", dtype={"id_hogar": str, "conglome": str})
datos = datos.merge(particion[["id_hogar", "conjunto", "pliegue"]], on="id_hogar", how="left", validate="one_to_one")
assert datos["conjunto"].notna().all(), "hay hogares sin partición"

# El diccionario dice qué columnas son predictores y de qué tipo.
dicc = pd.read_csv(PROCESSED / "diccionario_dataset_final.csv", encoding="utf-8-sig")

pred = dicc[dicc["grupo"] == "predictor"]
NUM = pred.loc[pred["tipo"] == "numérica", "variable"].tolist()
BIN = pred.loc[pred["tipo"] == "binaria (0/1)", "variable"].tolist()
CAT = pred.loc[pred["tipo"] == "categórica (código)", "variable"].tolist()
assert len(NUM) + len(BIN) + len(CAT) == len(pred), "hay predictores sin tipo reconocido en el diccionario"
assert set(NUM + BIN + CAT) <= set(datos.columns), "faltan columnas del diccionario en el dataset"
print(f"Predictores: {len(NUM)} numéricos, {len(BIN)} binarios, {len(CAT)} categóricos")

ent = datos[datos["conjunto"] == "entrenamiento"].reset_index(drop=True)
pru = datos[datos["conjunto"] == "prueba"].reset_index(drop=True)
y_ent, y_pru = ent["y"].to_numpy(), pru["y"].to_numpy()
print(f"Entrenamiento: {len(ent):,} hogares ({y_ent.mean():.1%} pobres) | Prueba: {len(pru):,} hogares ({y_pru.mean():.1%} pobres)")

## Paso 1. Nulos, codificación y escalado (el preprocesamiento)

**Por qué dentro del modelo y no antes:** el dataset conserva sus vacíos a propósito. Llenarlos exige calcular medianas y modas, escalar exige medias y desviaciones, y codificar exige conocer las categorías. Si esos números se calcularan con toda la base, la prueba influiría en cómo se preparan los datos de entrenamiento y la evaluación saldría demasiado optimista. Por eso el preprocesamiento se mete dentro de un `Pipeline`: en cada vuelta de la validación cruzada se ajusta **solo con los pliegues de entrenamiento de esa vuelta**, y en la evaluación final solo con el entrenamiento completo.

Cada tipo de variable se prepara por separado:

| Tipo | Qué se hace | Por qué |
|---|---|---|
| Numéricas | Vacíos con la **mediana** del entrenamiento; luego escalado (media 0, desviación 1). | La mediana resiste valores extremos. El escalado hace falta en KNN (distancias) y MLP, y no perjudica a la regresión logística. |
| Binarias (0/1) | Vacíos con la moda del entrenamiento. Sin escalar. | Ya están en una escala comparable. |
| Categóricas (códigos) | Vacíos como una categoría propia, `sin_dato`; luego **One-Hot**. | El código de una categoría no es una cantidad (el 4 no es "el doble" del 2), así que cada categoría pasa a ser una columna 0/1. El vacío tiene su propia columna en vez de inventar un valor. |

Los códigos de las categóricas se pasan a texto para que `4.0` y `4` sean la misma categoría.

In [ ]:
def codigos_a_texto(df, columnas):
    out = df[columnas].copy()
    for c in columnas:
        out[c] = out[c].map(lambda v: str(int(round(v))) if pd.notna(v) else np.nan).astype(object)
    return out


def preparar_X(df):
    return pd.concat([df[NUM + BIN].copy(), codigos_a_texto(df, CAT)], axis=1)


def construir_preprocesador():
    return ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), NUM),
        ("bin", SimpleImputer(strategy="most_frequent"), BIN),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="constant", fill_value="sin_dato")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT),
    ])


def con_preprocesamiento(modelo):
    return Pipeline([("pre", construir_preprocesador()), ("modelo", modelo)])


## Paso 1b. Revisar el preprocesamiento (solo verificación)

**Por qué:** antes de entrenar conviene ver qué hace el preprocesador. Aquí se ajusta una vez con todo el entrenamiento, **solo para revisarlo**: los modelos no usan este resultado, porque cada uno vuelve a ajustarlo dentro de cada pliegue. Se comprueba que no queden vacíos, que entrenamiento y prueba tengan las mismas columnas y que el escalado se vea como debe: en el entrenamiento las numéricas tienen media 0 y desviación 1; en la prueba se parecen pero **no** son exactamente 0 y 1 (señal de que no se ajustó con la prueba).

In [ ]:
X_ent, X_pru = preparar_X(ent), preparar_X(pru)
cv = PredefinedSplit(ent["pliegue"].to_numpy())          # los mismos 5 pliegues por conglomerado del cuaderno 04
print("Pliegues:", sorted(int(p) for p in ent["pliegue"].unique()))

revision = construir_preprocesador().fit(X_ent)           # solo entrenamiento
nombres = [n.split("__", 1)[1] for n in revision.get_feature_names_out()]
M_ent = pd.DataFrame(revision.transform(X_ent), columns=nombres)
M_pru = pd.DataFrame(revision.transform(X_pru), columns=nombres)
assert not M_ent.isna().any().any() and not M_pru.isna().any().any(), "quedan vacíos"
print(f"Sin vacíos. Columnas tras el preprocesamiento: {len(nombres)} | entrenamiento {M_ent.shape} | prueba {M_pru.shape}")

resumen = pd.DataFrame({"media_entrenamiento": M_ent[NUM].mean(), "desv_entrenamiento": M_ent[NUM].std(),
                        "media_prueba": M_pru[NUM].mean(), "desv_prueba": M_pru[NUM].std()}).round(3)
print(resumen.head(8).to_string())
sin_dato = [c for c in nombres if c.endswith("_sin_dato")]
print(f"\nColumnas 'sin_dato' creadas: {len(sin_dato)}")
print(M_ent[sin_dato].mean().sort_values(ascending=False).head(8).round(3).to_string())

## Paso 2. Métricas

**Por qué:** se definen una sola vez para usar exactamente las mismas en los cuatro modelos.
- **Sensibilidad** = pobres identificados / pobres reales. **Error de exclusión** = 1 - sensibilidad.
- **Precisión** = pobres identificados / hogares seleccionados. **Error de inclusión** = 1 - precisión.
- **A cobertura fija:** se seleccionan los hogares con mayor puntaje hasta completar el 20, 30 o 40 % de hogares. Es la forma de comparar modelos como si el programa tuviera un cupo fijo.
- **A umbral:** se seleccionan los hogares con puntaje mayor o igual al umbral elegido en validación cruzada.

### Ejemplo para leer las métricas
Ejemplo **ilustrativo** (no son tus resultados): en la prueba hay 6 700 hogares y 1 200 son pobres. El programa elige al 30 % con mayor puntaje, es decir 2 010 hogares, y 700 de ellos son pobres.
- **Sensibilidad** = 700 / 1 200 ≈ 58 %: el modelo encontró a 58 de cada 100 hogares pobres.
- **Error de exclusión** = 1 − 58 % ≈ 42 %: 500 hogares pobres quedaron fuera.
- **Precisión** = 700 / 2 010 ≈ 35 %: de cada 100 hogares seleccionados, 35 eran pobres.
- **Error de inclusión** = 1 − 35 % ≈ 65 %: 1 310 hogares seleccionados no eran pobres.
- **F1** es el promedio (armónico) de sensibilidad y precisión: solo es alto si ambas lo son.
- **PR-AUC** resume la relación entre precisión y sensibilidad para todos los umbrales posibles; no depende de un corte concreto.

Al subir la cobertura, la sensibilidad sube y la precisión baja. Por eso los modelos se comparan **a la misma cobertura**.

In [ ]:
def ordenar_por_puntaje(puntaje):
    """Índices de mayor a menor puntaje; los empates se resuelven al azar con semilla fija."""
    azar = np.random.default_rng(SEMILLA).random(len(puntaje))
    return np.lexsort((azar, -np.asarray(puntaje)))


def _metricas(y, seleccion):
    vp = int(y[seleccion].sum())
    k, p = int(len(seleccion)), int(y.sum())
    sens = vp / p if p else 0.0
    prec = vp / k if k else 0.0
    f1 = 2 * sens * prec / (sens + prec) if (sens + prec) else 0.0
    return {"hogares_seleccionados": k, "cobertura_real": k / len(y), "sensibilidad": sens, "error_exclusion": 1 - sens,
            "precision": prec, "error_inclusion": 1 - prec if k else np.nan, "f1": f1}


def metricas_cobertura(y, puntaje, cobertura):
    y = np.asarray(y)
    k = int(round(len(y) * cobertura))
    return _metricas(y, ordenar_por_puntaje(puntaje)[:k])


def metricas_umbral(y, puntaje, umbral):
    y = np.asarray(y)
    return _metricas(y, np.flatnonzero(np.asarray(puntaje) >= umbral))


def sensibilidad_a_cobertura(y, puntaje):
    return metricas_cobertura(y, puntaje, COBERTURA_AJUSTE)["sensibilidad"]


criterio = make_scorer(sensibilidad_a_cobertura, response_method="predict_proba")


def umbral_f1(y, puntaje):
    p, r, t = precision_recall_curve(y, puntaje)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
    return float(t[int(np.argmax(f1))])

## Paso 3. Los cuatro modelos y sus cuadrículas

**Por qué estos valores:** cada modelo tiene un hiperparámetro que controla su complejidad. Se prueba un conjunto pequeño y razonable de valores, de más simple a más complejo, para elegir con la validación cruzada y no a ojo.

| Modelo | Hiperparámetros que se prueban | Qué controlan |
|---|---|---|
| Regresión logística (L2) | `C` | Intensidad de la regularización (C pequeño = más regularización). |
| Árbol de decisión | `max_depth`, `min_samples_leaf` | Profundidad y tamaño mínimo de las hojas: árboles muy profundos memorizan el entrenamiento. |
| KNN | `n_neighbors`, `weights` | Cuántos vecinos votan y si los más cercanos pesan más. |
| MLP | `hidden_layer_sizes`, `alpha` | Tamaño de la red y regularización L2. |

### Los cuatro modelos en pocas palabras
| Modelo | Cómo decide | Fortaleza | Limitación |
|---|---|---|---|
| **Regresión logística** | Suma las características del hogar, cada una multiplicada por un coeficiente, y convierte el resultado en una probabilidad entre 0 y 1. | Simple e **interpretable**: los coeficientes dicen qué características se asocian con más o menos pobreza. | Solo capta relaciones en "línea recta" entre las características y la pobreza. |
| **Árbol de decisión** | Hace una serie de preguntas de sí o no sobre el hogar (por ejemplo, ¿el piso es de tierra?, ¿el jefe tiene primaria incompleta?) hasta llegar a una hoja con una probabilidad. | Se explica como reglas y capta combinaciones de características. | Un árbol muy profundo memoriza el entrenamiento (sobreajuste); por eso se limitan `max_depth` y `min_samples_leaf`. |
| **KNN** | Busca los K hogares de entrenamiento más parecidos y usa la proporción de pobres entre ellos. | No supone ninguna forma para la relación; es intuitivo. | Depende de la distancia: por eso se escala, y es sensible a variables poco útiles. No admite pesos de clase. |
| **MLP (red neuronal)** | Capas de unidades que combinan las características y aprenden relaciones no lineales. | Puede captar patrones complejos. | Más difícil de interpretar y más sensible a sus hiperparámetros. |

No hay un ganador seguro de antemano. En datos de este tipo (tabulares, con muchas variables categóricas), los modelos sencillos a menudo quedan cerca de los complejos. Medir cuánto, si es que mejoran, es justamente lo que compara este trabajo.

In [ ]:
if RAPIDO:
    GRILLAS = {
        "Regresión logística": {"modelo__C": [0.01, 1]},
        "Árbol de decisión": {"modelo__max_depth": [5, 10], "modelo__min_samples_leaf": [50]},
        "KNN": {"modelo__n_neighbors": [31, 101], "modelo__weights": ["uniform"]},
        "MLP": {"modelo__hidden_layer_sizes": [(32,)], "modelo__alpha": [1e-3]},
    }
else:
    GRILLAS = {
        "Regresión logística": {"modelo__C": [0.001, 0.01, 0.1, 1, 10]},
        "Árbol de decisión": {"modelo__max_depth": [3, 5, 7, 10, 15], "modelo__min_samples_leaf": [20, 50, 100]},
        "KNN": {"modelo__n_neighbors": [15, 31, 51, 101], "modelo__weights": ["uniform", "distance"]},
        "MLP": {"modelo__hidden_layer_sizes": [(32,), (64,), (64, 32)], "modelo__alpha": [1e-4, 1e-2]},
    }

MODELOS = {
    "Regresión logística": LogisticRegression(class_weight="balanced", max_iter=3000),   # regularización L2 (por defecto)
    "Árbol de decisión": DecisionTreeClassifier(class_weight="balanced", random_state=SEMILLA),
    "KNN": KNeighborsClassifier(),
    "MLP": MLPClassifier(early_stopping=True, max_iter=300, random_state=SEMILLA),
}
PESOS_ENT = compute_sample_weight("balanced", y_ent)     # solo para el MLP (KNN no admite pesos)

## Paso 4. Entrenar, elegir y evaluar

**Por qué en este orden, para cada modelo:**
1. **Búsqueda en cuadrícula** con la validación cruzada por conglomerado → mejores hiperparámetros (por sensibilidad al 30 %). Al terminar reentrena con todo el entrenamiento.
2. **Predicciones de validación** de cada hogar de entrenamiento cuando estuvo fuera del pliegue de entrenamiento → umbral que maximiza F1. Se hace con el modelo ya elegido, otra vez dentro de cada pliegue.
3. **Prueba (una sola vez):** puntajes de los hogares de prueba y métricas con el umbral y a cobertura fija.

La prueba no se usa para elegir nada. Con los datos completos, el MLP puede tardar varios minutos.

In [ ]:
resultados_cv, resultados_prueba, predicciones, mejores = [], [], {"id_hogar": pru["id_hogar"].values, "y": y_pru}, {}
modelos_ajustados = {}

for nombre, modelo in MODELOS.items():
    t0 = time.time()
    extra = {"modelo__sample_weight": PESOS_ENT} if nombre == "MLP" else {}

    gs = GridSearchCV(con_preprocesamiento(modelo), GRILLAS[nombre], scoring=criterio, cv=cv, n_jobs=N_JOBS, refit=True)
    gs.fit(X_ent, y_ent, **extra)
    i = gs.best_index_
    mejores[nombre] = {k.replace("modelo__", ""): (list(v) if isinstance(v, tuple) else v) for k, v in gs.best_params_.items()}
    resultados_cv.append({"modelo": nombre, "mejores_parametros": json.dumps(mejores[nombre], ensure_ascii=False),
                          f"sensibilidad_al_{int(COBERTURA_AJUSTE*100)}pct_cv_media": gs.cv_results_["mean_test_score"][i],
                          f"sensibilidad_al_{int(COBERTURA_AJUSTE*100)}pct_cv_desv": gs.cv_results_["std_test_score"][i]})

    oof = cross_val_predict(gs.best_estimator_, X_ent, y_ent, cv=cv, method="predict_proba", n_jobs=N_JOBS,
                            params=extra)[:, 1]
    umbral = umbral_f1(y_ent, oof)

    puntaje = gs.predict_proba(X_pru)[:, 1]
    predicciones[nombre] = puntaje
    modelos_ajustados[nombre] = gs.best_estimator_
    fila = {"modelo": nombre, "pr_auc": average_precision_score(y_pru, puntaje)}
    resultados_prueba.append({**fila, "criterio": f"umbral F1 (CV) = {umbral:.3f}", **metricas_umbral(y_pru, puntaje, umbral)})
    for c in COBERTURAS:
        resultados_prueba.append({**fila, "criterio": f"cobertura {int(c*100)} %", **metricas_cobertura(y_pru, puntaje, c)})

    print(f"{nombre}: {mejores[nombre]} | sensibilidad al {int(COBERTURA_AJUSTE*100)} % en CV = "
          f"{resultados_cv[-1][f'sensibilidad_al_{int(COBERTURA_AJUSTE*100)}pct_cv_media']:.3f} | {time.time() - t0:.0f} s")

resultados_cv = pd.DataFrame(resultados_cv)
resultados_prueba = pd.DataFrame(resultados_prueba)

## Paso 5. Resultados

**Por qué dos tablas:** la de validación cruzada muestra qué tan bien funcionó cada modelo con los datos de entrenamiento y cuánto varía entre pliegues (la desviación estándar). La de prueba es la evaluación final, con hogares que no se usaron para elegir nada.

En la tabla de prueba, para el **error de exclusión** menor es mejor; mira sobre todo las filas a cobertura fija, que comparan los modelos con el mismo cupo.

In [ ]:
pd.set_option("display.width", 200)
print("VALIDACIÓN CRUZADA (entrenamiento)")
display(resultados_cv.round(4))

print("PRUEBA (evaluación final)")
cols = ["modelo", "criterio", "hogares_seleccionados", "sensibilidad", "error_exclusion", "precision", "error_inclusion", "f1", "pr_auc"]
display(resultados_prueba[cols].round(4))

print("Error de exclusión en prueba, por cobertura (menor es mejor):")
tabla = resultados_prueba[resultados_prueba["criterio"].str.startswith("cobertura")].pivot(index="modelo", columns="criterio", values="error_exclusion")
display(tabla.round(4))

### Cómo leer estas tablas
1. **Compáralas con el azar.** Un modelo sin información que eligiera al 30 % de los hogares encontraría **cerca del 30 %** de los pobres (sensibilidad ≈ 0,30) y su precisión sería **cerca de la tasa de pobreza** (≈ 18 %). Su PR-AUC también sería ≈ 0,18. Un modelo útil debe estar claramente por encima de esos valores.
2. **La tabla de error de exclusión por cobertura es la clave para tu pregunta de investigación** (menor es mejor). Mira si el orden de los modelos se mantiene en 20, 30 y 40 %. Si cambia, la conclusión depende de cuántos hogares pueda atender el programa.
3. **Desviación estándar entre pliegues** (tabla de validación cruzada): si es del tamaño de la diferencia entre dos modelos, esa diferencia puede ser ruido. No afirmes que un modelo es mejor por una diferencia pequeña.
4. **Validación cruzada frente a prueba:** deben ser parecidas. Compara la sensibilidad al 30 % de la validación cruzada con la fila "cobertura 30 %" de la prueba. Si la prueba es mucho peor, hay sobreajuste o la partición no es representativa.
5. **Las filas "umbral F1" no se comparan solas entre modelos.** Mira cuántos hogares seleccionó cada modelo (`hogares_seleccionados`): seleccionar más hogares aumenta la sensibilidad casi siempre, a costa de la precisión. Por eso la comparación justa es a cobertura fija.
6. **Señales de alerta:** sensibilidad muy alta con precisión muy baja (el modelo selecciona casi a todos), un PR-AUC casi igual a la tasa de pobreza (casi no aprende) o un modelo que gana en una cobertura y pierde en las otras.

## Paso 6. Interpretar lo que aprendió cada modelo

**Por qué:** además de qué modelo acierta más, el informe necesita decir qué características pesan. En la regresión logística se miran los coeficientes (positivo = más probabilidad de pobreza); en el árbol, la importancia de cada variable. Los nombres vienen de las columnas del preprocesador (las categóricas aparecen una vez por categoría).

In [ ]:
def nombres_columnas(pipe):
    return [n.split("__", 1)[1] for n in pipe.named_steps["pre"].get_feature_names_out()]


lr = modelos_ajustados["Regresión logística"]
coef = pd.Series(lr.named_steps["modelo"].coef_[0], index=nombres_columnas(lr)).sort_values()
print("Regresión logística: 10 características que MÁS REDUCEN la probabilidad de pobreza")
print(coef.head(10).round(3).to_string())
print("\n10 que MÁS LA AUMENTAN")
print(coef.tail(10)[::-1].round(3).to_string())

arbol = modelos_ajustados["Árbol de decisión"]
imp = pd.Series(arbol.named_steps["modelo"].feature_importances_, index=nombres_columnas(arbol)).sort_values(ascending=False)
print("\nÁrbol de decisión: 10 variables más importantes")
print(imp.head(10).round(3).to_string())

### Cómo interpretar coeficientes e importancias
- **Signo en la regresión logística:** un coeficiente **positivo** significa que esa característica, con las demás iguales, **aumenta** la probabilidad de que el hogar sea pobre; uno **negativo** la **reduce**. Las numéricas están escaladas, así que sus coeficientes se pueden comparar entre sí. Una columna One-Hot (por ejemplo `p103_5`) indica "el hogar pertenece a esa categoría" frente a no pertenecer.
- **Es asociación, no causa.** Que tener TV por cable aparezca con coeficiente negativo no significa que regalar televisores reduzca la pobreza: es un indicador del nivel de vida del hogar.
- **Variables parecidas se reparten el efecto.** Si dos variables dicen casi lo mismo (por ejemplo, el material del piso y el del techo), el modelo divide el efecto entre ellas. Un coeficiente pequeño no prueba que la característica sea irrelevante.
- **Importancia en el árbol:** mide cuánto ayudó esa variable a separar pobres de no pobres en las divisiones del árbol. Suma 1 en total y **no dice el signo** (no distingue si aumenta o reduce la pobreza). Una variable puede salir con poca importancia porque otra parecida ya hizo ese trabajo.
- **Columnas `sin_dato` o "no aplica":** si alguna aparece entre las más influyentes, significa que el patrón de ausencia es informativo (por ejemplo, ser o no propietario de la vivienda).
- **Para la parte ética del informe:** características como el área, el tipo de vivienda o la composición del hogar pueden reflejar desigualdades existentes. Por eso más adelante se revisan los errores por departamento, área y lengua materna.

## Paso 7. Guardar

**Por qué:** el siguiente cuaderno (análisis de errores por departamento, área urbana o rural y lengua) necesita las predicciones sobre la prueba. Los mejores parámetros y las tablas se citan en el informe.

In [ ]:
resultados_cv.to_csv(SALIDA / "resultados_cv.csv", index=False, encoding="utf-8-sig")
resultados_prueba.to_csv(SALIDA / "resultados_prueba.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(predicciones).to_csv(SALIDA / "predicciones_prueba.csv", index=False, encoding="utf-8-sig")
with open(SALIDA / "mejores_parametros.json", "w", encoding="utf-8") as f:
    json.dump(mejores, f, ensure_ascii=False, indent=2)
joblib.dump(modelos_ajustados, SALIDA / "modelos_ajustados.joblib")
print("Guardado en", SALIDA)